# Exercice 1 : CNN Variante VGG pour CIFAR-10
## TP2 — Réseaux de Neurones Convolutifs

---

**Objectif :** Construire et entraîner un CNN inspiré de VGG pour classifier des images CIFAR-10.

---

## Étape 1 — Activer le GPU

**Runtime → Change runtime type → GPU**

In [ ]:
import torch
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## Étape 2 — Imports

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print(f"TensorFlow : {tf.__version__}")
print(f"GPU : {tf.config.list_physical_devices('GPU')}")

---
## Étape 3 — Charger CIFAR-10

**Dataset :**
- 60 000 images couleur 32×32
- 10 classes : avion, voiture, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion

In [ ]:
# Charger les données
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()

print(f"Train : {train_images.shape}")
print(f"Test : {test_images.shape}")

In [ ]:
# Afficher quelques images
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i][0]])
    plt.axis('off')
plt.tight_layout()
plt.show()

## Étape 4 — Prétraitement

In [ ]:
# Normaliser (0-1)
train_images, test_images = train_images / 255.0, test_images / 255.0

# Séparer validation (20%)
train_images, val_images, train_labels, val_labels = train_test_split(
    train_images, train_labels, test_size=0.2, random_state=42
)

print(f"Train : {train_images.shape}")
print(f"Validation : {val_images.shape}")
print(f"Test : {test_images.shape}")

---
## Étape 5 — Construire le modèle VGG-like

**Architecture :**
```
Bloc 1 : Conv(32) → Conv(32) → MaxPool → Dropout
Bloc 2 : Conv(64) → Conv(64) → MaxPool → Dropout
Bloc 3 : Conv(128) → Conv(128) → MaxPool → Dropout
FC : Flatten → Dense(512) → Dropout → Dense(10)
```

In [ ]:
model = models.Sequential([
    # Bloc 1
    layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)),
    layers.Conv2D(32, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Bloc 2
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.Conv2D(64, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Bloc 3
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Classification
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

## Étape 6 — Compiler et entraîner

In [ ]:
# Compiler
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Entraîner
history = model.fit(
    train_images, train_labels,
    epochs=20,
    batch_size=64,
    validation_data=(val_images, val_labels)
)

## Étape 7 — Évaluer

In [ ]:
# Évaluation sur le test set
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f"\nTest accuracy : {test_acc:.2%}")

## Étape 8 — Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Résumé

| Composant | Rôle |
|-----------|------|
| Conv2D | Extraction de features |
| MaxPooling | Réduction de taille |
| Dropout | Régularisation (évite le surapprentissage) |
| Dense | Classification finale |
| Softmax | Probabilités pour chaque classe |